# DX 704 Week 10 Project

In this project, you will implement document search within a question and answer database and assess its performance.


The full project description and a template notebook are available on GitHub: [Project 10 Materials](https://github.com/bu-cds-dx704/dx704-project-10).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [18]:
# =====================
# Important Libraries
# =====================

import json

import numpy as np 
import pandas as pd

import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer

## Part 1: Download the SQuAD-explorer Data Set

You may use the code provided below.

In [1]:
!git clone https://github.com/rajpurkar/SQuAD-explorer

Cloning into 'SQuAD-explorer'...
remote: Enumerating objects: 5563, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 5563 (delta 11), reused 17 (delta 6), pack-reused 5539 (from 1)
Receiving objects: 100% (5563/5563), 52.26 MiB | 36.55 MiB/s, done.
Resolving deltas: 100% (3563/3563), done.


In [2]:
import json

In [3]:
with open("SQuAD-explorer/dataset/train-v1.1.json") as fp:
    train_data = json.load(fp)

In [4]:
type(train_data)

dict

In [5]:
list(train_data.keys())

['data', 'version']

In [6]:
type(train_data["data"])

list

In [7]:
len(train_data["data"])

442

In [8]:
type(train_data["data"][0])

dict

In [9]:
train_data["data"][0].keys()

dict_keys(['title', 'paragraphs'])

In [10]:
train_data["data"][0]["title"]

'University_of_Notre_Dame'

In [11]:
len(train_data["data"][0]["paragraphs"])

55

In [12]:
train_data["data"][0]["paragraphs"][0]

{'context': 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.',
 'qas': [{'answers': [{'answer_start': 515,
     'text': 'Saint Bernadette Soubirous'}],
   'question': 'To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?',
   'id': '5733be284776f41900661182'},
  {'answers': [{'answer_start': 188, 'text': 'a copper statue of Christ

In [14]:
sum(len(doc["paragraphs"]) for doc in train_data["data"])

18896

## Part 2: Restructure JSON Data for Processing

Parse the file "SQuAD-explorer/dataset/train-v1.1.json" above to produce a file "parsed.tsv" with columns document_title, paragraph_index, and paragraph_context.
The paragraph_index column should be zero-indexed, so zero for the first paragraph of each document.
Use pandas `to_csv` method to write the file since there are many quotes and other issues to handle otherwise.

In [22]:
# =============================================
# Part 2: Restructure JSON Data for Processing
# =============================================

with open('SQuAD-explorer/dataset/train-v1.1.json') as fp:
    squad_data = json.load(fp)

rows = []
for document in squad_data["data"]:
    for paragraph_index, paragraph in enumerate(document['paragraphs']):
        rows.append({
            'document_title': document['title'],
            'paragraph_index': paragraph_index,  # zero-indexed per document
            'paragraph_context': paragraph['context'],
        })


# =====================================
# Save to TSV file named 'parsed.tsv'
# =====================================

parsed_df = pd.DataFrame(rows, columns=['document_title', 'paragraph_index', 'paragraph_context'])
parsed_df.to_csv('parsed.tsv', sep='\t', index=False)
parsed_df

,document_title,paragraph_index,paragraph_context
0,University_of_Notre_Dame,0,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,1,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,2,The university is the major seat of the Congre...
3,University_of_Notre_Dame,3,The College of Engineering was established in ...
4,University_of_Notre_Dame,4,All of Notre Dame's undergraduate students are...
...,...,...,...
18891,Kathmandu,53,"Institute of Medicine, the central college of ..."
18892,Kathmandu,54,Football and Cricket are the most popular spor...
18893,Kathmandu,55,The total length of roads in Nepal is recorded...
18894,Kathmandu,56,The main international airport serving Kathman...


Submit "parsed.tsv" in Gradescope.

## Part 3: Prepare Suitable Paragraph Vectors for Document Search

Design and implement paragraph vectors based on their text with length 1024.
Note that this will be much smaller than the number of distinct words in the training data.

Hint: you can base your vectors on any techniques covered in this module so far.
Beware that they will be automatically assessed (along with the question vectors of part 4) to make sure they retain useful information.

In [23]:
# ===============================================================
# Part 3: Prepare Suitable Paragraph Vectors for Document Search
# ===============================================================

vector_size = 1024

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(parsed_df["paragraph_context"])

rng = np.random.default_rng(704)
n_words = X_tfidf.shape[1]
word_bucket = rng.integers(0, vector_size, size=n_words)
word_sign = rng.integers(0, 2, size=n_words) * 2 - 1
word_hash = sp.csr_matrix(
    (word_sign, (np.arange(n_words), word_bucket)),
    shape=(n_words, vector_size),
    )

def normalize_features(X):
    norms = np.sqrt(np.asarray(X.multiply(X).sum(axis=1))).ravel()
    norms[norms == 0] = 1  # avoid dividing by zero for empty texts
    return sp.diags(1 / norms) @ X

def encode_texts(texts):
    return normalize_features(tfidf_vectorizer.transform(texts) @ word_hash).toarray()

paragraph_vectors = encode_texts(parsed_df["paragraph_context"])
paragraph_vectors.shape  # (18896, 1024)

(18896, 1024)

Save your paragraph vectors in a file "paragraph-vectors.tsv.gz" with columns document_title, paragraph_index, and paragraph_vector_json where paragraph_vector_json is a JSON encoded list.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

In [24]:
# ====================================================================
# Save paragraph vectors to TSV file named 'paragraph_vectors.tsv.gz'
# ====================================================================

paragraph_vectors_df = pd.DataFrame({
    'document_title': parsed_df["document_title"],
    'paragraph_index': parsed_df["paragraph_index"],
    'paragraph_vector_json': [json.dumps(vector.tolist()) for vector in paragraph_vectors],
    })

# Save file
paragraph_vectors_df.to_csv("paragraph_vectors.tsv.gz", sep="\t", index=False)
paragraph_vectors_df.head()

,document_title,paragraph_index,paragraph_vector_json
0,University_of_Notre_Dame,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,University_of_Notre_Dame,1,"[0.0, 0.0, 0.0, -0.2632487564978246, 0.0, 0.0,..."
2,University_of_Notre_Dame,2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,University_of_Notre_Dame,3,"[0.0, 0.0, 0.0, 0.0, 0.08188594839640191, 0.0,..."
4,University_of_Notre_Dame,4,"[0.0, 0.0, 0.0, -0.08872514578951451, 0.0, 0.0..."


Submit "paragraph-vectors.tsv.gz" in Gradescope.

## Part 4: Encode Question Vectors with the Same Design

Read the questions in "questions.tsv" and encode them in the same way that you encoded the paragraph vectors.

In [25]:
# ===================================================
# Part 4: Encode Question Vectors with Same Design
# ===================================================

# Read questions in 'questions.tsv' and encode them using the same vector design

questions_df = pd.read_csv('questions.tsv', sep='\t')
question_vectors = encode_texts(questions_df["question"])
question_vectors.shape

(100, 1024)

Save your question vectors in "question-vectors.tsv" with columns question_id and question_vector_json.

In [26]:
# ===============================================================
# Save Question Vectors to TSV File named 'question_vectors.tsv'
# =============================================================== 

question_vectors_df = pd.DataFrame({
    'question_id': questions_df["question_id"],
    'question_vector_json': [json.dumps(vector.tolist()) for vector in question_vectors],
    })

# Save file
question_vectors_df.to_csv("question-vectors.tsv", sep="\t", index=False)
question_vectors_df.head()

,question_id,question_vector_json
0,1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,4,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,7,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,10,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,13,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


Submit "question-vectors.tsv" in Gradescope.

## Part 5: Match Questions to Paragraphs using Nearest Neighbors

Match your question vectors to paragraph vectors and identify the top 5 paragraph vectors for each question using nearest neighbors.
Specifically, use the Euclidean distance between the vectors.


In [27]:
# ===============================================================
# Part 5: Match Questions to Paragraphs Using Nearest Neighbors
# ===============================================================

from sklearn.neighbors import NearestNeighbors

# Fit Nearest Neighbors model on paragraph vectors
nn_model = NearestNeighbors(n_neighbors=5, metric="euclidean")
nn_model.fit(paragraph_vectors)

# Find 5 nearest paragraph for each question
distances, indices = nn_model.kneighbors(question_vectors)

Save your top matches in a file "question-matches.tsv" with columns question_id, question_rank, document_title, and paragraph_index.


In [30]:
# ============================================================
# Save top matches to TSV file named 'question-matches.tsv'
# ============================================================

rows = []
for question_id, neighbor_rows in zip(questions_df["question_id"], indices):
    for question_rank, row in enumerate(neighbor_rows, start=1):
        rows.append({
            "question_id": question_id,
            "question_rank": question_rank,  # 1 = closest match
            "document_title": parsed_df["document_title"].iloc[row],
            "paragraph_index": parsed_df["paragraph_index"].iloc[row],
        })

question_matches_df = pd.DataFrame(rows)

# Save file
question_matches_df.to_csv("question-matches.tsv", sep="\t", index=False)
question_matches_df.head(15)

,question_id,question_rank,document_title,paragraph_index
0,1,1,Buddhism,47
1,1,2,Federal_Bureau_of_Investigation,19
2,1,3,Buddhism,138
3,1,4,Buddhism,98
4,1,5,Association_football,21
5,4,1,BeiDou_Navigation_Satellite_System,18
6,4,2,BeiDou_Navigation_Satellite_System,13
7,4,3,BeiDou_Navigation_Satellite_System,14
8,4,4,BeiDou_Navigation_Satellite_System,12
9,4,5,BeiDou_Navigation_Satellite_System,7


Submit "question-matches.tsv" in Gradescope.

## Part 6: Spot Check Question and Paragraph Matches

Review the paragraphs matched to the first 5 questions (sorted by question_id ascending).
Which paragraph was the worst match for each question?


Submit "worst-paragraphs.tsv" in Gradescope.

Write a file "worst-paragraphs.tsv" with three columns question_id, document_title, paragraph_index.

In [33]:
# ====================================================
#  Part 6: Spot Check Question and Paragraph Matches
# ====================================================

# Display top 5 question matches for spot checking
question_matches_df.head(5)

# Find paragraph that was worst match for each question
worst_paragraphs_df1 = question_matches_df[question_matches_df["question_rank"] == question_matches_df["question_rank"].max()]
worst_paragraphs_df1.head(5)

,question_id,question_rank,document_title,paragraph_index
4,1,5,Association_football,21
9,4,5,BeiDou_Navigation_Satellite_System,7
14,7,5,Kanye_West,22
19,10,5,Roman_Republic,47
24,13,5,"Tucson,_Arizona",23


In [35]:
# ==============================================================
# Submit worst matches to TSV file named 'worst-paragraphs.tsv'
# ==============================================================

# Use columns 'question_id', 'document_title', and 'paragraph_index'
worst_paragraphs_df2 = worst_matches[["question_id", "document_title", "paragraph_index"]]

# Save file
worst_paragraphs_df2.to_csv("worst-paragraphs.tsv", sep="\t", index=False)
worst_paragraphs_df2

,question_id,document_title,paragraph_index
4,1,Association_football,21
9,4,BeiDou_Navigation_Satellite_System,7
14,7,Kanye_West,22
19,10,Roman_Republic,47
24,13,"Tucson,_Arizona",23
...,...,...,...
479,286,British_Empire,26
484,289,Mexico_City,31
489,292,North_Carolina,23
494,295,Neptune,28


## Part 7: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 8: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.